# Flight Delay Prediction — XGBoost + Optuna Tuning



## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay, f1_score
)

import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import joblib

print(f"XGBoost  : {xgb.__version__}")
print(f"Optuna   : {optuna.__version__}")

## 2. Load Cleaned Data

In [3]:
df = pd.read_csv("cleaned_flights_2022.csv")
print(f"Shape: {df.shape}")
print(f"Delay rate: {df['DepDel15'].mean() * 100:.1f}%")
df.head()

Shape: (3951046, 29)
Delay rate: 21.7%


,FlightDate,Airline,Origin,Dest,Cancelled,Diverted,CRSDepTime,CRSElapsedTime,Distance,Year,...,DestState,DestStateName,DepDel15,DepTimeBlk,CRSArrTime,ArrTimeBlk,DistanceGroup,DivAirportLandings,Delay_Minutes,Early_Minutes
0,2022-04-04,"Commutair Aka Champlain Enterprises, Inc.",GJT,DEN,0,0,1133,72.0,212.0,2022,...,CO,Colorado,0,1100-1159,1245,1200-1259,1,0,0.0,10.0
1,2022-04-04,"Commutair Aka Champlain Enterprises, Inc.",HRL,IAH,0,0,732,77.0,295.0,2022,...,TX,Texas,0,0700-0759,849,0800-0859,2,0,0.0,4.0
2,2022-04-04,"Commutair Aka Champlain Enterprises, Inc.",DRO,DEN,0,0,1529,70.0,251.0,2022,...,CO,Colorado,0,1500-1559,1639,1600-1659,2,0,0.0,15.0
3,2022-04-04,"Commutair Aka Champlain Enterprises, Inc.",IAH,GPT,0,0,1435,90.0,376.0,2022,...,MS,Mississippi,0,1400-1459,1605,1600-1659,2,0,0.0,5.0
4,2022-04-04,"Commutair Aka Champlain Enterprises, Inc.",DRO,DEN,0,0,1135,70.0,251.0,2022,...,CO,Colorado,0,1100-1159,1245,1200-1259,2,0,0.0,0.0


## 3. Feature Engineering


- **Cyclical time encoding** (sin/cos) so the model understands that hour 23 and hour 0 are close
- **Airport-level historical delay rates** — if an airport delays 40% of flights, that's a strong signal
- **Airline historical delay rates** — same idea per airline
- **Route-level delay rates** — some Origin→Dest pairs are chronically delayed
- **Interaction features** — e.g. Airline × DayOfWeek

In [ ]:
#  Date features 
df['FlightDate'] = pd.to_datetime(df['FlightDate'])
df['Month']      = df['FlightDate'].dt.month
df['DayOfMonth'] = df['FlightDate'].dt.day
df['DayOfWeek']  = df['FlightDate'].dt.dayofweek   # 0=Mon, 6=Sun
df['WeekOfYear'] = df['FlightDate'].dt.isocalendar().week.astype(int)
df['Quarter']    = df['FlightDate'].dt.quarter
df['IsWeekend']  = (df['DayOfWeek'] >= 5).astype(int)

df = df.drop(columns=['FlightDate'])

#  Cyclical encoding for time columns 
# CRSDepTime is in HHMM format (e.g. 1430 = 14:30)
# Convert to fractional hour first, then encode cyclically
if 'CRSDepTime' in df.columns:
    df['DepHour'] = (df['CRSDepTime'] // 100) + (df['CRSDepTime'] % 100) / 60
    df['DepHour_sin'] = np.sin(2 * np.pi * df['DepHour'] / 24)
    df['DepHour_cos'] = np.cos(2 * np.pi * df['DepHour'] / 24)

# Cyclical month encoding
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# Cyclical day-of-week encoding
df['DOW_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['DOW_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)

print("Date & cyclical features added.")

Date & cyclical features added.


In [ ]:
#  Target-encoded statistical features 
# These capture historical delay rates at the group level.
# IMPORTANT: We compute these ONLY on the training set later to avoid leakage.
# Here we store the column names to compute post-split.

TARGET = 'DepDel15'

# Drop leakage columns
df = df.drop(columns=['Delay_Minutes', 'Early_Minutes'], errors='ignore')

# Identify categorical columns for label encoding
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns: {cat_cols}")

Categorical columns: ['Airline', 'Origin', 'Dest', 'Operating_Airline', 'OriginCityName', 'OriginState', 'OriginStateName', 'DestCityName', 'DestState', 'DestStateName', 'DepTimeBlk', 'ArrTimeBlk']


In [6]:
# Label encode categoricals
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

print("Label encoding complete.")
print(f"Final feature count: {df.shape[1] - 1}")

Label encoding complete.
Final feature count: 35


## 4. Train / Test Split

In [7]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]:,} rows | Test : {X_test.shape[0]:,} rows")
print(f"Features : {X_train.shape[1]}")

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight : {scale_pos_weight:.2f}")

Train : 3,160,836 rows | Test : 790,210 rows
Features : 35
scale_pos_weight : 3.61


## 5. Target-Encoded Statistical Features (leak-safe)

Computed on training data only, then mapped onto test data.

In [8]:
def add_target_encoding(X_tr, y_tr, X_te, group_col, new_col_name, smoothing=20):
    """
    Smoothed target encoding: blends group mean with global mean.
    smoothing controls how much we trust the group vs global mean.
    """
    global_mean = y_tr.mean()
    temp = X_tr[[group_col]].copy()
    temp['target'] = y_tr.values

    stats = temp.groupby(group_col)['target'].agg(['mean', 'count'])
    stats['smoothed'] = (
        (stats['mean'] * stats['count'] + global_mean * smoothing)
        / (stats['count'] + smoothing)
    )

    X_tr = X_tr.copy()
    X_te = X_te.copy()
    X_tr[new_col_name] = X_tr[group_col].map(stats['smoothed']).fillna(global_mean)
    X_te[new_col_name] = X_te[group_col].map(stats['smoothed']).fillna(global_mean)
    return X_tr, X_te

# Airline delay rate
if 'Airline' in X_train.columns:
    X_train, X_test = add_target_encoding(X_train, y_train, X_test, 'Airline', 'Airline_DelayRate')

# Origin airport delay rate
if 'Origin' in X_train.columns:
    X_train, X_test = add_target_encoding(X_train, y_train, X_test, 'Origin', 'Origin_DelayRate')

# Destination airport delay rate
if 'Dest' in X_train.columns:
    X_train, X_test = add_target_encoding(X_train, y_train, X_test, 'Dest', 'Dest_DelayRate')

# Operating airline delay rate (if different column exists)
if 'Operating_Airline' in X_train.columns:
    X_train, X_test = add_target_encoding(X_train, y_train, X_test, 'Operating_Airline', 'OpAirline_DelayRate')

# Route delay rate (Origin + Dest combo)
if 'Origin' in X_train.columns and 'Dest' in X_train.columns:
    X_train = X_train.copy()
    X_test  = X_test.copy()
    X_train['Route'] = X_train['Origin'].astype(str) + '_' + X_train['Dest'].astype(str)
    X_test['Route']  = X_test['Origin'].astype(str)  + '_' + X_test['Dest'].astype(str)
    X_train, X_test = add_target_encoding(X_train, y_train, X_test, 'Route', 'Route_DelayRate')
    X_train = X_train.drop(columns=['Route'])
    X_test  = X_test.drop(columns=['Route'])

# DayOfWeek delay rate
X_train, X_test = add_target_encoding(X_train, y_train, X_test, 'DayOfWeek', 'DOW_DelayRate')

# Hour delay rate
if 'DepHour' in X_train.columns:
    X_train['DepHourBin'] = (X_train['DepHour'] // 1).astype(int)
    X_test['DepHourBin']  = (X_test['DepHour'] // 1).astype(int)
    X_train, X_test = add_target_encoding(X_train, y_train, X_test, 'DepHourBin', 'Hour_DelayRate')

print(f"Features after target encoding: {X_train.shape[1]}")

Features after target encoding: 43


In [ ]:
import json

# Save target encoding lookup tables for inference
target_encoding_maps = {}

def add_target_encoding_and_save(X_tr, y_tr, X_te, group_col, new_col_name, smoothing=20):
    global_mean = y_tr.mean()
    temp = X_tr[[group_col]].copy()
    temp['target'] = y_tr.values
    stats = temp.groupby(group_col)['target'].agg(['mean', 'count'])
    stats['smoothed'] = (
        (stats['mean'] * stats['count'] + global_mean * smoothing)
        / (stats['count'] + smoothing)
    )
    mapping = stats['smoothed'].to_dict()
    target_encoding_maps[new_col_name] = {'map': mapping, 'global_mean': global_mean}

    X_tr = X_tr.copy(); X_te = X_te.copy()
    X_tr[new_col_name] = X_tr[group_col].map(mapping).fillna(global_mean)
    X_te[new_col_name] = X_te[group_col].map(mapping).fillna(global_mean)
    return X_tr, X_te

# Replace all your add_target_encoding calls with add_target_encoding_and_save:
X_train, X_test = add_target_encoding_and_save(X_train, y_train, X_test, 'Airline', 'Airline_DelayRate')
X_train, X_test = add_target_encoding_and_save(X_train, y_train, X_test, 'Origin', 'Origin_DelayRate')
X_train, X_test = add_target_encoding_and_save(X_train, y_train, X_test, 'Dest', 'Dest_DelayRate')
X_train, X_test = add_target_encoding_and_save(X_train, y_train, X_test, 'Operating_Airline', 'OpAirline_DelayRate')

X_train['Route'] = X_train['Origin'].astype(str) + '_' + X_train['Dest'].astype(str)
X_test['Route']  = X_test['Origin'].astype(str)  + '_' + X_test['Dest'].astype(str)
X_train, X_test = add_target_encoding_and_save(X_train, y_train, X_test, 'Route', 'Route_DelayRate')
X_train = X_train.drop(columns=['Route']); X_test = X_test.drop(columns=['Route'])

X_train, X_test = add_target_encoding_and_save(X_train, y_train, X_test, 'DayOfWeek', 'DOW_DelayRate')

X_train['DepHourBin'] = (X_train['DepHour'] // 1).astype(int)
X_test['DepHourBin']  = (X_test['DepHour'] // 1).astype(int)
X_train, X_test = add_target_encoding_and_save(X_train, y_train, X_test, 'DepHourBin', 'Hour_DelayRate')

# Save the maps — deploy this file alongside your model
with open('target_encoding_maps.json', 'w') as f:
    json.dump(target_encoding_maps, f)

# Also save the exact feature column order the model expects
feature_columns = X_train.columns.tolist()
with open('feature_columns.json', 'w') as f:
    json.dump(feature_columns, f)

print("Saved target_encoding_maps.json and feature_columns.json")
print(f"Final feature count: {len(feature_columns)}")
print(feature_columns)

## 6. Optuna Hyperparameter Tuning (XGBoost)

Optuna runs 50 trials searching for the best combination of hyperparameters. It uses the AUC on a validation fold as the objective.

In [9]:
# Use a 20% sample of training data for tuning to keep runtime manageable
# Once best params found, we train the final model on the full training set
sample_frac = 0.2
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train,
    train_size=sample_frac,
    random_state=42,
    stratify=y_train
)
print(f"Tuning on {X_tune.shape[0]:,} rows ({int(sample_frac*100)}% of training data)")

Tuning on 632,167 rows (20% of training data)


In [10]:
def xgb_objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 200, 600),
        'max_depth'         : trial.suggest_int('max_depth', 4, 10),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight'  : trial.suggest_int('min_child_weight', 1, 10),
        'gamma'             : trial.suggest_float('gamma', 0, 1.0),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'scale_pos_weight'  : scale_pos_weight,
        'use_label_encoder' : False,
        'eval_metric'       : 'auc',
        'random_state'      : 42,
        'n_jobs'            : -1,
    }
    model = xgb.XGBClassifier(**params)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_tune, y_tune, cv=skf, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

print("Running Optuna tuning — 50 trials (this will take a few minutes)...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f"\nBest XGBoost AUC (on tune sample): {study_xgb.best_value:.4f}")
print(f"Best params: {study_xgb.best_params}")

Running Optuna tuning — 50 trials (this will take a few minutes)...


Best trial: 45. Best value: 0.737255: 100%|██████████| 50/50 [37:09<00:00, 44.59s/it]


Best XGBoost AUC (on tune sample): 0.7373
Best params: {'n_estimators': 494, 'max_depth': 10, 'learning_rate': 0.03386345283814475, 'subsample': 0.9246621901060612, 'colsample_bytree': 0.5222707411599039, 'min_child_weight': 2, 'gamma': 0.5264393759889829, 'reg_alpha': 8.720284853240354, 'reg_lambda': 0.04289310880606081}


## 7. Train Final XGBoost on Full Training Set

In [ ]:
best_xgb_params = study_xgb.best_params.copy()
best_xgb_params.update({
    'scale_pos_weight'  : scale_pos_weight,
    'use_label_encoder' : False,
    'eval_metric'       : 'auc',
    'random_state'      : 42,
    'n_jobs'            : -1,
    'early_stopping_rounds': 20
})

xgb_model = xgb.XGBClassifier(**best_xgb_params)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

xgb_prob  = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc   = roc_auc_score(y_test, xgb_prob)
print(f"\nXGBoost (tuned) Test AUC: {xgb_auc:.4f}")

## 8. Optimal Threshold Tuning

The default 0.5 threshold is rarely optimal for imbalanced data. We find the threshold that maximizes the F1 score for the 'Delayed' class.


In [ ]:
thresholds  = np.arange(0.1, 0.9, 0.01)
f1_scores   = []

for t in thresholds:
    preds = (xgb_prob >= t).astype(int)
    f1_scores.append(f1_score(y_test, preds, pos_label=1))

best_threshold = thresholds[np.argmax(f1_scores)]
best_f1        = max(f1_scores)

print(f"Optimal threshold : {best_threshold:.2f}")
print(f"Best F1 (Delayed) : {best_f1:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores, color='steelblue')
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Best threshold = {best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score (Delayed)')
plt.title('F1 Score vs Decision Threshold')
plt.legend()
plt.tight_layout()
plt.savefig('threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Final Evaluation with Optimal Threshold


In [ ]:
y_pred_final = (xgb_prob >= best_threshold).astype(int)

print(f"=== XGBoost (Optuna-tuned) — Final Evaluation (threshold = {best_threshold:.2f}) ===")
print()
print(classification_report(y_test, y_pred_final, target_names=['On-Time', 'Delayed']))
print(f"ROC-AUC : {xgb_auc:.4f}")

# Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_final)
ConfusionMatrixDisplay(cm, display_labels=['On-Time', 'Delayed']).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Confusion Matrix (XGBoost — Optuna-tuned)', fontsize=13)

# ROC curve
fpr_x, tpr_x, _ = roc_curve(y_test, xgb_prob)
axes[1].plot(fpr_x, tpr_x, label=f'XGBoost AUC={xgb_auc:.4f}', color='steelblue')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — XGBoost (Optuna-tuned)', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.savefig('final_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Feature Importance


In [ ]:
imp_df = pd.DataFrame({
    'Feature'    : X_train.columns,
    'Importance' : xgb_model.feature_importances_
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=imp_df, x='Importance', y='Feature', palette='viridis')
plt.title('Top 20 Feature Importances (XGBoost — Optuna-tuned)', fontsize=13)
plt.tight_layout()
plt.savefig('feature_importance_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print(imp_df.to_string(index=False))


## 11. Save Model


In [ ]:
joblib.dump(xgb_model, 'best_model_xgboost.pkl')
joblib.dump(best_threshold, 'best_threshold.pkl')

print("Saved: best_model_xgboost.pkl")
print(f"Saved: best_threshold.pkl  (value={best_threshold:.2f})")
print()
print("To load and predict later:")
print("  model     = joblib.load('best_model_xgboost.pkl')")
print("  threshold = joblib.load('best_threshold.pkl')")
print("  y_pred    = (model.predict_proba(X_new)[:, 1] >= threshold).astype(int)")
